# HVAC Predictive Maintenance & Energy Efficiency

Este notebook analiza lecturas operativas de equipos HVAC para detectar riesgo de falla en los próximos 7 días.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score, accuracy_score

In [ ]:
df = pd.read_csv('../data/hvac_equipment_readings.csv')
df.head()

In [ ]:
df.info()
df.describe()

## Distribución de la variable objetivo

La variable `failure_next_7_days` indica si el equipo presentó riesgo de falla en los próximos 7 días.

In [ ]:
df['failure_next_7_days'].value_counts(normalize=True)

## Análisis exploratorio

In [ ]:
df.groupby('failure_next_7_days')[[
    'delta_t_f','head_pressure_psig','compressor_amps','vibration_ips',
    'filter_pressure_drop_inwc','energy_kwh'
]].mean()

In [ ]:
plt.figure(figsize=(8,5))
df.groupby('failure_next_7_days')['delta_t_f'].mean().plot(kind='bar')
plt.title('Promedio de Delta T por condición de falla')
plt.xlabel('Falla en próximos 7 días')
plt.ylabel('Delta T promedio (°F)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.scatter(df['outdoor_temp_f'], df['energy_kwh'], alpha=0.5)
plt.title('Relación entre temperatura exterior y consumo energético')
plt.xlabel('Temperatura exterior (°F)')
plt.ylabel('Consumo energético (kWh)')
plt.tight_layout()
plt.show()

## Preparación de datos y entrenamiento del modelo

In [ ]:
features = ['unit_age_years','outdoor_temp_f','return_air_temp_f','supply_air_temp_f','delta_t_f','suction_pressure_psig','head_pressure_psig','compressor_amps','vibration_ips','filter_pressure_drop_inwc','runtime_hours','energy_kwh']
X = df[features].fillna(df[features].median(numeric_only=True))
y = df['failure_next_7_days']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
model = RandomForestClassifier(n_estimators=250, random_state=42, class_weight='balanced', max_depth=8)
model.fit(X_train, y_train)
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

## Evaluación del modelo

In [ ]:
print(classification_report(y_test, pred))
print('Accuracy:', round(accuracy_score(y_test, pred), 3))
print('F1-score:', round(f1_score(y_test, pred), 3))
print('AUC-ROC:', round(roc_auc_score(y_test, proba), 3))

In [ ]:
cm = confusion_matrix(y_test, pred)
plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title('Matriz de confusión')
plt.xlabel('Predicción')
plt.ylabel('Valor real')
for (i, j), val in np.ndenumerate(cm):
    plt.text(j, i, int(val), ha='center', va='center')
plt.xticks([0,1], ['No falla', 'Falla'])
plt.yticks([0,1], ['No falla', 'Falla'])
plt.tight_layout()
plt.show()

## Importancia de variables

In [ ]:
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
importances.head(10)

In [ ]:
plt.figure(figsize=(8,5))
importances.head(8).sort_values().plot(kind='barh')
plt.title('Variables más importantes del modelo')
plt.xlabel('Importancia')
plt.tight_layout()
plt.show()

## Conclusión

El análisis muestra que variables como Delta T, presión de descarga, amperaje, vibración, presión diferencial del filtro y consumo energético pueden ayudar a identificar equipos con mayor probabilidad de falla. Este tipo de modelo puede apoyar decisiones de mantenimiento preventivo y priorización de inspecciones.